# 03_1 — Human validation analysis

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 03_1 — Human validation analysis

Analyzes the two reviewers’ completed independent blinded ratings. The final manuscript reports inter-rater agreement and reviewer-specific stratum/cutoff summaries; no consensus adjudication is used.


In [ ]:
# ============================================================
# 1. SETUP
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

VALIDATION_DIR = PROJECT_ROOT / "validation"
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "human_validation_analysis"
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Public repository default omits post text and contains only the variables needed
# to reproduce the reported agreement and stratum-level statistics.
RESULTS_FILE = VALIDATION_DIR / "threshold_pair_validation_ratings_only.csv"

STRATUM_ORDER = [
    "0.60_to_lt_0.65",
    "0.65_to_lt_0.70",
    "0.70_to_lt_0.75",
    "0.75_to_0.85",
]
CANDIDATE_CUTOFFS = [0.65, 0.70, 0.75]

print("Validation input file:", RESULTS_FILE)
print("Analysis output folder:", ANALYSIS_OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. SAFE OUTPUT HELPERS
# ============================================================
def versioned_path(path):
    path = Path(path)
    if not path.exists():
        return path
    i = 1
    while True:
        candidate = path.with_name(f"{path.stem}_rerun{i:02d}{path.suffix}")
        if not candidate.exists():
            return candidate
        i += 1

def safe_save_csv(dataframe, filename):
    out = versioned_path(ANALYSIS_OUTPUT_DIR / filename)
    dataframe.to_csv(out, index=False)
    print("Saved:", out)
    return out


In [ ]:
# ============================================================
# 3. LOAD COMPLETED HUMAN RATINGS + BASIC QC
# ============================================================
if not RESULTS_FILE.exists():
    raise FileNotFoundError(f"Missing completed ratings file: {RESULTS_FILE}")

df = pd.read_csv(RESULTS_FILE)
print("Loaded completed ratings file:", RESULTS_FILE)

required_cols = [
    "pair_id", "cosine_similarity", "similarity_stratum",
    "reviewer_1_similar", "reviewer_2_similar"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError("Missing required columns: " + ", ".join(missing_cols))

for col in ["reviewer_1_similar", "reviewer_2_similar"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Shape:", df.shape)
print("Missing ratings:")
display(df[["reviewer_1_similar", "reviewer_2_similar"]].isna().sum().to_frame("missing_n"))

if df["pair_id"].duplicated().any():
    raise ValueError("Duplicate pair_id values detected.")
if len(df) != 200:
    print(f"WARNING: expected 200 validation pairs, found {len(df)}.")
if df[["reviewer_1_similar", "reviewer_2_similar"]].isna().any().any():
    raise ValueError("At least one reviewer rating is missing.")
for col in ["reviewer_1_similar", "reviewer_2_similar"]:
    observed = set(df[col].unique())
    if not observed.issubset({0, 1}):
        raise ValueError(f"{col} contains values other than 0/1: {observed}")

print("Pairs per similarity stratum:")
display(df["similarity_stratum"].value_counts().reindex(STRATUM_ORDER).rename("n_pairs").to_frame())
print("Reviewer rating distributions:")
display(pd.DataFrame({
    "Reviewer 1": df["reviewer_1_similar"].value_counts().sort_index(),
    "Reviewer 2": df["reviewer_2_similar"].value_counts().sort_index(),
}))


In [ ]:
# ============================================================
# 4. INTER-RATER AGREEMENT
# ============================================================
agreement_mask = df["reviewer_1_similar"] == df["reviewer_2_similar"]
n_pairs = len(df)
n_agree = int(agreement_mask.sum())
n_disagree = int((~agreement_mask).sum())
percent_agreement = n_agree / n_pairs
kappa = cohen_kappa_score(df["reviewer_1_similar"], df["reviewer_2_similar"])

agreement_table = pd.crosstab(
    df["reviewer_1_similar"], df["reviewer_2_similar"],
    rownames=["Reviewer 1"], colnames=["Reviewer 2"], margins=True
)
agreement_summary = pd.DataFrame([{
    "n_pairs": n_pairs,
    "n_agree": n_agree,
    "n_disagree": n_disagree,
    "percent_agreement": percent_agreement,
    "cohens_kappa": kappa,
}])

print("INTER-RATER AGREEMENT")
display(agreement_summary)
display(agreement_table)
print(f"Percent agreement: {percent_agreement:.1%}")
print(f"Cohen's kappa: {kappa:.3f}")
print(f"Disagreements: {n_disagree}/{n_pairs}")

safe_save_csv(agreement_summary, "human_validation_interrater_summary.csv")


In [ ]:
# ============================================================
# 5A. HUMAN SIMILARITY RATE BY STRATUM
# ============================================================
rows = []
for stratum in STRATUM_ORDER:
    sub = df[df["similarity_stratum"] == stratum].copy()
    if len(sub) == 0:
        continue
    r1_rate = sub["reviewer_1_similar"].mean()
    r2_rate = sub["reviewer_2_similar"].mean()
    rows.append({
        "similarity_stratum": stratum,
        "n_pairs": len(sub),
        "reviewer_1_n_similar": int(sub["reviewer_1_similar"].sum()),
        "reviewer_1_rate": r1_rate,
        "reviewer_2_n_similar": int(sub["reviewer_2_similar"].sum()),
        "reviewer_2_rate": r2_rate,
        "mean_reviewer_rate": (r1_rate + r2_rate) / 2,
    })
stratum_pre = pd.DataFrame(rows)
print("RESULTS BY STRATUM")
display(stratum_pre.style.format({
    "reviewer_1_rate": "{:.1%}",
    "reviewer_2_rate": "{:.1%}",
    "mean_reviewer_rate": "{:.1%}",
}))
safe_save_csv(stratum_pre, "human_validation_by_stratum.csv")


In [ ]:
# ============================================================
# 5B. CUTOFF COMPARISON
# ============================================================
cutoff_rows = []
for cutoff in CANDIDATE_CUTOFFS:
    below = df[df["cosine_similarity"] < cutoff]
    above = df[df["cosine_similarity"] >= cutoff]
    for reviewer in ["reviewer_1_similar", "reviewer_2_similar"]:
        below_rate = below[reviewer].mean()
        above_rate = above[reviewer].mean()
        cutoff_rows.append({
            "cutoff": cutoff,
            "reviewer": reviewer,
            "n_below": len(below),
            "below_similar_rate": below_rate,
            "n_at_or_above": len(above),
            "at_or_above_similar_rate": above_rate,
            "difference_percentage_points": (above_rate - below_rate) * 100,
        })
cutoff_pre = pd.DataFrame(cutoff_rows)
print("CUTOFF COMPARISON")
display(cutoff_pre.style.format({
    "below_similar_rate": "{:.1%}",
    "at_or_above_similar_rate": "{:.1%}",
    "difference_percentage_points": "{:.1f}",
}))
safe_save_csv(cutoff_pre, "human_validation_cutoff.csv")
print("0.70 cutoff, by reviewer:")
display(cutoff_pre[cutoff_pre["cutoff"] == 0.70].style.format({
    "below_similar_rate": "{:.1%}",
    "at_or_above_similar_rate": "{:.1%}",
    "difference_percentage_points": "{:.1f}",
}))


## Independent-reviewer visualization

This figure provides a descriptive visual check of whether human-rated semantic similarity increases across the prespecified cosine-similarity strata using the two independent blinded ratings. Reviewer-specific rates and their descriptive mean are shown.


In [ ]:
# ============================================================
# 5C. INDEPENDENT-REVIEWER VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt

plot_df = stratum_pre.copy()

label_map = {
    "0.60_to_lt_0.65": "0.60–<0.65",
    "0.65_to_lt_0.70": "0.65–<0.70",
    "0.70_to_lt_0.75": "0.70–<0.75",
    "0.75_to_0.85": "0.75–0.85",
}

plot_df["stratum_label"] = plot_df["similarity_stratum"].map(label_map)
x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    x,
    plot_df["reviewer_1_rate"] * 100,
    marker="o",
    label="Reviewer 1",
)
ax.plot(
    x,
    plot_df["reviewer_2_rate"] * 100,
    marker="o",
    label="Reviewer 2",
)
ax.plot(
    x,
    plot_df["mean_reviewer_rate"] * 100,
    marker="o",
    linewidth=2.5,
    label="Mean of reviewers",
)

# The selected 0.70 threshold lies between the second and third strata.
ax.axvline(1.5, linestyle="--", linewidth=1.2)
ax.text(
    1.53,
    96,
    "Selected threshold = 0.70",
    rotation=90,
    va="top",
    ha="left",
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["stratum_label"])
ax.set_xlabel("Cosine-similarity stratum")
ax.set_ylabel("Pairs judged semantically similar (%)")
ax.set_ylim(0, 100)
ax.set_title("Human-rated semantic similarity across cosine-similarity strata")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

figure_path = ANALYSIS_OUTPUT_DIR / "human_validation_similarity_by_stratum.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", figure_path)


## Metrics reported in the revised manuscript

The final analysis uses the two reviewers’ independent blinded ratings without consensus adjudication. Report: (1) 200 pairs, 50 per prespecified similarity stratum; (2) percent agreement; (3) Cohen’s kappa; (4) reviewer-specific similarity proportions across the four strata; and (5) reviewer-specific proportions below versus at/above 0.70.
